# LRPD Wine Price Analysis
**Analyst:** Juan Xavier Gomez Illingworth

**NB:** Parts of the code were constructed with the assistance of Gen AI.

In [29]:
# Load analysis libraries
import pandas as pd
import json
import numpy as np
import altair as alt

In [30]:
# Load LRPD prices dataset and preview
prices = pd.read_parquet('https://autocpi-public.s3.eu-west-2.amazonaws.com/lrpd/db_prices.parquet', engine='fastparquet')
prices.head()

,quote_date,shop_code,item_id_raw,region,price,indicator_box,item_id
0,200102.0,808.0,210101,12.0,0.35,Q,210101
1,199603.0,32.0,210101,3.0,0.42,,210101
2,198905.0,3.0,210101,8.0,0.44,,210101
3,199511.0,52.0,210101,2.0,0.64,,210101
4,200105.0,126.0,210101,8.0,0.80,,210101


In [31]:
# Load LRPD item metadata and preview
items = pd.read_parquet('https://autocpi-public.s3.eu-west-2.amazonaws.com/lrpd/db_item.parquet', engine='fastparquet')
items.head()

,item_id,description,date_quote_s,date_quote_e,n_obs
0,210101,LARGE LOAF-WHITE-SLICED-800G,198802,200401,36039
1,210102,LARGE LOAF-WHITE-UNSLICED-800G,198802,202510,56917
2,210105,LARGE WHOLEMEAL LOAF-UNSLICED,198802,200301,27161
3,210106,SIX BREAD ROLLS-WHITE/BROWN,198802,202510,67469
4,210107,"BROWN LOAF,400G,SLICED-GRAN",198903,200401,29361


In [32]:
# Load date dimension table and preview
dates = pd.read_csv('../data/db_date.csv')
dates.head()

,date,quote_date,obs_panel,obs,date2,date3,date4,year,year_s,month,...,p_med,p_sd,day,day_s,cpiindex00allitems2015100,cpiindex01foodandnonalcoholicbev,cpiindex02alcoholicbeveragestoba,cpiindex011food2015100,cpiindex012nonalcoholicbeverages,cpi00_adjustment
0,1,198802,35459.0,51065,01-02-1988,01-02-1988,1988-02-01,1988,1988,2,...,1.35,54.338814,1,1,48.6,48.3,27.4,48.5,45.9,1.000000
1,2,198803,35240.0,49987,01-03-1988,01-03-1988,1988-03-01,1988,1988,3,...,1.30,51.615620,1,1,48.7,48.4,27.5,48.6,46.2,1.002058
2,3,198804,36066.0,51716,01-04-1988,01-04-1988,1988-04-01,1988,1988,4,...,1.35,53.485882,1,1,49.3,48.7,27.9,48.9,46.6,1.014403
3,4,198805,35869.0,51209,01-05-1988,01-05-1988,1988-05-01,1988,1988,5,...,1.35,53.936596,1,1,49.5,48.8,28.0,49.0,47.2,1.018519
4,5,198806,35811.0,51126,01-06-1988,01-06-1988,1988-06-01,1988,1988,6,...,1.38,53.684380,1,1,49.7,48.9,28.0,49.0,47.7,1.022634


In [33]:
# Load item weights and preview
weights = pd.read_csv('../data/db_itemWeights.csv')
weights.head()

,quote_date,item_id,coicop_weight,dup
0,199602.0,210101.0,2.75,0
1,199603.0,210101.0,2.75,0
2,199604.0,210101.0,2.75,0
3,199605.0,210101.0,2.75,0
4,199606.0,210101.0,2.75,0


In [34]:
# Load region lookup and preview
regions = pd.read_csv('../data/db_region.csv')
regions.head()

,region_n,region_s,region,country,obs,p_min,p_max,p_mean,p_med,p_sd
0,1,Catalogue collections,Catalogue,NaN,306968,0.12,1999,41.329037,22.00,80.908554
1,2,London,London,England,4475848,0.01,20000,56.248341,4.60,246.010300
2,3,South East,South East,England,5785278,0.01,9499,54.609386,4.90,220.251300
3,4,South West,South West,England,3597289,0.01,6950,50.904106,4.50,201.636340
4,5,East Anglia,East Anglia,England,3204009,0.01,7650,51.067471,4.69,197.143280


In [35]:
# Identify wine-related items
items[items['description'].str.contains('wine', case=False)].head()

,item_id,description,date_quote_s,date_quote_e,n_obs
470,310304,WINE (PER GLASS),198802,200301,108862
475,310310,"WINE, PER 175 - 250 ML SERVING",200302,202510,110404
479,310315,BOTTLE OF WINE 70-75CL,199402,202510,142720
486,310406,FORTIFIED WINE (70-75CL),198802,202510,95368
488,310409,OTHER IMPTD WHITE WINE-70-75CL,198802,200401,38031


In [36]:
# Define wine category-to-item mapping
wine_mapping = pd.DataFrame({
    'wine_category': ['RED WINE 70-75 CL'] * 4 + ['ROSE WINE 70-75 CL'] + ['WHITE WINE 70-75 CL'] * 3,
    'item_id': [310412, 310421, 310422, 310431, 310425, 310409, 310419, 310420]
})
wine_mapping

,wine_category,item_id
0,RED WINE 70-75 CL,310412
1,RED WINE 70-75 CL,310421
2,RED WINE 70-75 CL,310422
3,RED WINE 70-75 CL,310431
4,ROSE WINE 70-75 CL,310425
5,WHITE WINE 70-75 CL,310409
6,WHITE WINE 70-75 CL,310419
7,WHITE WINE 70-75 CL,310420


In [37]:
# Compute red wine price quantiles by month
red_wine_ids = wine_mapping[wine_mapping['wine_category'] == 'RED WINE 70-75 CL']['item_id'].tolist()
red_wine_prices = prices[prices['item_id'].isin(red_wine_ids)]
red_wine_stats = red_wine_prices.groupby('quote_date').agg({
    'price': ['median', lambda x: x.quantile(0.10), lambda x: x.quantile(0.25), lambda x: x.quantile(0.75), lambda x: x.quantile(0.90)]
}).reset_index()
red_wine_stats.columns = ['date', 'Median', 'P10', 'Q1', 'Q3', 'P90']
red_wine_stats.head()

,date,Median,P10,Q1,Q3,P90
0,198802.0,2.35,1.796,1.99,2.7000,2.990
1,198803.0,2.39,1.890,1.99,2.8275,3.047
2,198804.0,2.45,1.890,1.99,2.8900,3.090
3,198805.0,2.45,1.858,1.99,2.8900,3.098
4,198806.0,2.45,1.890,2.05,2.8900,3.190


In [38]:
# Compute rose wine price quantiles by month
rose_wine_ids = wine_mapping[wine_mapping['wine_category'] == 'ROSE WINE 70-75 CL']['item_id'].tolist()
rose_wine_prices = prices[prices['item_id'].isin(rose_wine_ids)]
rose_wine_stats = rose_wine_prices.groupby('quote_date').agg({
    'price': ['median', lambda x: x.quantile(0.10), lambda x: x.quantile(0.25), lambda x: x.quantile(0.75), lambda x: x.quantile(0.90)]
}).reset_index()
rose_wine_stats.columns = ['date', 'Median', 'P10', 'Q1', 'Q3', 'P90']
rose_wine_stats.head()

,date,Median,P10,Q1,Q3,P90
0,200902.0,5.350,3.99,4.4900,6.0050,6.988
1,200903.0,4.990,3.99,4.4900,5.8500,6.876
2,200904.0,5.185,3.99,4.5825,5.9900,6.990
3,200905.0,5.065,3.99,4.2900,6.2100,6.990
4,200906.0,5.065,3.99,4.4650,6.2275,6.990


In [39]:
# Compute white wine price quantiles by month
white_wine_ids = wine_mapping[wine_mapping['wine_category'] == 'WHITE WINE 70-75 CL']['item_id'].tolist()
white_wine_prices = prices[prices['item_id'].isin(white_wine_ids)]
white_wine_stats = white_wine_prices.groupby('quote_date').agg({
    'price': ['median', lambda x: x.quantile(0.10), lambda x: x.quantile(0.25), lambda x: x.quantile(0.75), lambda x: x.quantile(0.90)]
}).reset_index()
white_wine_stats.columns = ['date', 'Median', 'P10', 'Q1', 'Q3', 'P90']
white_wine_stats.head()

,date,Median,P10,Q1,Q3,P90
0,198802.0,2.19,1.79,1.95,2.7925,3.09
1,198803.0,2.19,1.79,1.95,2.8500,3.19
2,198804.0,2.19,1.79,1.99,2.8450,3.19
3,198805.0,2.19,1.85,1.99,2.8900,3.09
4,198806.0,2.23,1.85,1.99,2.8900,3.05


In [40]:
# Combine wine stats, add dates, and preview
red_wine_stats['wine_type'] = 'RED WINE 70-75 CL'
rose_wine_stats['wine_type'] = 'ROSE WINE 70-75 CL'
white_wine_stats['wine_type'] = 'WHITE WINE 70-75 CL'
all_wine_stats = pd.concat([red_wine_stats, rose_wine_stats, white_wine_stats], ignore_index=True)
all_wine_stats['date_str'] = all_wine_stats['date'].astype(int).astype(str)
all_wine_stats['year'] = all_wine_stats['date_str'].str[:4].astype(int)
all_wine_stats['month'] = all_wine_stats['date_str'].str[4:6].astype(int)
all_wine_stats['datetime'] = pd.to_datetime(all_wine_stats[['year', 'month']].assign(day=1))
all_wine_stats.head()

,date,Median,P10,Q1,Q3,P90,wine_type,date_str,year,month,datetime
0,198802.0,2.35,1.796,1.99,2.7000,2.990,RED WINE 70-75 CL,198802,1988,2,1988-02-01
1,198803.0,2.39,1.890,1.99,2.8275,3.047,RED WINE 70-75 CL,198803,1988,3,1988-03-01
2,198804.0,2.45,1.890,1.99,2.8900,3.090,RED WINE 70-75 CL,198804,1988,4,1988-04-01
3,198805.0,2.45,1.858,1.99,2.8900,3.098,RED WINE 70-75 CL,198805,1988,5,1988-05-01
4,198806.0,2.45,1.890,2.05,2.8900,3.190,RED WINE 70-75 CL,198806,1988,6,1988-06-01


In [41]:
# Build interactive quantile bands and median chart
wine_selector = alt.selection_point(
    name='wine_type',
    fields=['wine_type'],
    bind=alt.binding_select(
        options=['RED WINE 70-75 CL', 'ROSE WINE 70-75 CL', 'WHITE WINE 70-75 CL'],
        name='Pick a wine type: '
    ),
    value='RED WINE 70-75 CL'
)
filtered_data = all_wine_stats
area_iqr = alt.Chart(filtered_data).transform_calculate(
    iqr_color="datum.wine_type == 'RED WINE 70-75 CL' ? '#8B1A1A' : datum.wine_type == 'ROSE WINE 70-75 CL' ? '#C75B7A' : '#D4AF37'"
).mark_area(opacity=0.8).encode(
    x=alt.X('datetime:T', title='Year', axis=alt.Axis(format='%Y')),
    y=alt.Y('Q1:Q', title='Price (£)', scale=alt.Scale(domain=[0, 12])),
    y2='Q3:Q',
    color=alt.Color('iqr_color:N', scale=None, legend=None),
    tooltip=[
        alt.Tooltip('datetime:T', title='Date', format='%B %Y'),
        alt.Tooltip('wine_type:N', title='Wine Type'),
        alt.Tooltip('Median:Q', title='Median Price (£)', format=',.2f'),
        alt.Tooltip('Q1:Q', title='Q1 (25th percentile)', format=',.2f'),
        alt.Tooltip('Q3:Q', title='Q3 (75th percentile)', format=',.2f'),
        alt.Tooltip('P10:Q', title='P10 (10th percentile)', format=',.2f'),
        alt.Tooltip('P90:Q', title='P90 (90th percentile)', format=',.2f')
    ]
).add_params(
    wine_selector
).transform_filter(
    wine_selector
)
area_p10_q1 = alt.Chart(filtered_data).transform_calculate(
    p10_q1_color="datum.wine_type == 'RED WINE 70-75 CL' ? '#C85C5C' : datum.wine_type == 'ROSE WINE 70-75 CL' ? '#E8A0BF' : '#F4D03F'"
).mark_area(opacity=0.5).encode(
    x='datetime:T',
    y=alt.Y('P10:Q', scale=alt.Scale(domain=[0, 12])),
    y2='Q1:Q',
    color=alt.Color('p10_q1_color:N', scale=None, legend=None),
    tooltip=[
        alt.Tooltip('datetime:T', title='Date', format='%B %Y'),
        alt.Tooltip('wine_type:N', title='Wine Type'),
        alt.Tooltip('Median:Q', title='Median Price (£)', format=',.2f'),
        alt.Tooltip('Q1:Q', title='Q1 (25th percentile)', format=',.2f'),
        alt.Tooltip('Q3:Q', title='Q3 (75th percentile)', format=',.2f'),
        alt.Tooltip('P10:Q', title='P10 (10th percentile)', format=',.2f'),
        alt.Tooltip('P90:Q', title='P90 (90th percentile)', format=',.2f')
    ]
).transform_filter(
    wine_selector
)
area_q3_p90 = alt.Chart(filtered_data).transform_calculate(
    q3_p90_color="datum.wine_type == 'RED WINE 70-75 CL' ? '#C85C5C' : datum.wine_type == 'ROSE WINE 70-75 CL' ? '#E8A0BF' : '#F4D03F'"
).mark_area(opacity=0.5).encode(
    x='datetime:T',
    y=alt.Y('Q3:Q', scale=alt.Scale(domain=[0, 12])),
    y2='P90:Q',
    color=alt.Color('q3_p90_color:N', scale=None, legend=None),
    tooltip=[
        alt.Tooltip('datetime:T', title='Date', format='%B %Y'),
        alt.Tooltip('wine_type:N', title='Wine Type'),
        alt.Tooltip('Median:Q', title='Median Price (£)', format=',.2f'),
        alt.Tooltip('Q1:Q', title='Q1 (25th percentile)', format=',.2f'),
        alt.Tooltip('Q3:Q', title='Q3 (75th percentile)', format=',.2f'),
        alt.Tooltip('P10:Q', title='P10 (10th percentile)', format=',.2f'),
        alt.Tooltip('P90:Q', title='P90 (90th percentile)', format=',.2f')
    ]
).transform_filter(
    wine_selector
)
line_median = alt.Chart(filtered_data).transform_calculate(
    median_color="datum.wine_type == 'RED WINE 70-75 CL' ? '#5C0A0A' : datum.wine_type == 'ROSE WINE 70-75 CL' ? '#9B4159' : '#B8860B'"
).mark_line(strokeWidth=2).encode(
    x='datetime:T',
    y=alt.Y('Median:Q', scale=alt.Scale(domain=[0, 12])),
    color=alt.Color('median_color:N', scale=None, legend=None),
    tooltip=[
        alt.Tooltip('datetime:T', title='Date', format='%B %Y'),
        alt.Tooltip('wine_type:N', title='Wine Type'),
        alt.Tooltip('Median:Q', title='Median Price (£)', format=',.2f'),
        alt.Tooltip('Q1:Q', title='Q1 (25th percentile)', format=',.2f'),
        alt.Tooltip('Q3:Q', title='Q3 (75th percentile)', format=',.2f'),
        alt.Tooltip('P10:Q', title='P10 (10th percentile)', format=',.2f'),
        alt.Tooltip('P90:Q', title='P90 (90th percentile)', format=',.2f')
    ]
).transform_filter(
    wine_selector
)
chart = (area_p10_q1 + area_iqr + area_q3_p90 + line_median).properties(
    width=800,
    height=500,
    title={
        'text': 'LRPD Wine Price Distribution History',
        'subtitle': 'Source: Long Run Prices Database (LRPD). Davies (2021).',
        'anchor': 'start'
    }
)
chart

alt.LayerChart(...)

In [42]:
# Save the chart as JSON
chart.save('../graphs/lrpd_wine_price_history.json')